# 1.3 TCN 튜닝

- **비교** · 4초/24채널 AdamW → 20초/24채널 AdamW → 20초/48채널 AdamW → 20초/48채널 Adam.
- **기준** · 기존 4초/24채널 AdamW, 고정 6 epoch. 새 후보는 학습률 0.002, 가중치 감쇠 0.001, 최대 20 epoch.
- **조기 종료** · 바깥 검증 경기 1개를 빼고, 남은 6경기 중 1개로 종료 시점을 고른다. 선택한 epoch 수로 6경기를 다시 학습한 뒤 바깥 경기에서 점수를 낸다.
- **실행** · Python 3.12, 같은 폴더의 `dfl_model_comparison.ipynb`와 그 노트북의 공통 환경 사용. 셀을 위에서부터 실행한다.
- **출력** · `model/영상추출/추출csv/dead_ball/dfl_tcn_tuning_20261111_01/`.
- **옵티마이저 비교** · 이 설정에서 Adam과 AdamW의 가중치 감쇠 방식 차이를 비교한다. 모델·배치·학습률·시드를 맞춘다.

In [ ]:
import json
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/train_dfl.py").is_file()
)
folder = root / "model/영상추출/모델/dead_ball"
shared = folder / "dfl_model_comparison.ipynb"
shared_cells = {cell["id"]: cell for cell in json.loads(shared.read_text())["cells"]}
for cell_id in ("dfl13-2", "dfl13-4", "dfl13-6"):
    source = "".join(shared_cells[cell_id]["source"])
    exec(compile(source, f"{shared}:{cell_id}", "exec"))

import torch
from torch import nn

BASELINE_OUTPUT = OUT / "model_summary.csv"
NOTEBOOK = folder / "dfl_tcn_tuning.ipynb"
RUN_ID = "dfl_tcn_tuning_20261111_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
MAX_EPOCHS = 20
MIN_EPOCHS = 4
CHECK_EVERY = 2
PATIENCE = 3
LEARNING_RATE = 0.002
WEIGHT_DECAY = 0.001
TORCH_SEED = 17
torch.set_num_threads(min(THREADS, 4))
CANDIDATES = {
    "4s 24 AdamW": dict(window_s=4, width=24, optimizer="AdamW", dilations=(1, 2, 3, 4)),
    "20s 24 AdamW": dict(window_s=20, width=24, optimizer="AdamW", dilations=(1, 3, 8, 13, 25)),
    "20s 48 AdamW": dict(window_s=20, width=48, optimizer="AdamW", dilations=(1, 3, 8, 13, 25)),
    "20s 48 Adam": dict(window_s=20, width=48, optimizer="Adam", dilations=(1, 3, 8, 13, 25)),
}

## 학습과 조기 종료

- 선수 특징·샘플링·정규화·채점은 기존 TCN과 같다.
- 4 epoch부터 2 epoch마다 내부 경기의 비터비 적용 후 인플레이 구간 F1을 확인한다. 3번 연속 개선이 없으면 종료한다.
- 전체 20초 창은 기준 프레임 앞뒤 10초다. 학습 시점 간격 0.2초 기준 101개 시점이 시간 합성곱의 범위에 들어간다.

In [ ]:
def build_tcn(spec):
    width = spec["width"]
    layers = [nn.Conv1d(len(BASE) * 2, width, 1), nn.GELU()]
    for dilation in spec["dilations"]:
        layers.extend([nn.Conv1d(width, width, 3, dilation=dilation), nn.GELU()])
    layers.append(nn.Conv1d(width, 1, 1))
    return nn.Sequential(*layers)


def make_training_data(matches, train, radius):
    matrix = np.concatenate([train[match][0][:, BASE_INDEX] for match in matches])
    mean = np.nanmean(matrix, axis=0)
    scale = np.maximum(np.nanstd(matrix, axis=0), 1e-3)
    assert np.isfinite(mean).all() and np.isfinite(scale).all()

    examples, labels = [], []
    for match in matches:
        x, y, period = train[match]
        values = normalized(x[:, BASE_INDEX], mean, scale)
        for half in period_slices(period):
            examples.append(tcn_chunks(values[half], radius))
            targets = y[half].astype(np.float32)
            padded = np.pad(targets, (0, (-len(targets)) % CHUNK), constant_values=-1)
            labels.append(padded.reshape(-1, CHUNK))
    return torch.from_numpy(np.concatenate(examples)), torch.from_numpy(np.concatenate(labels)), mean, scale


def predict_tcn(model, values, period, mean, scale, radius):
    model.eval()
    score = np.empty(len(values), dtype=np.float32)
    values = normalized(values, mean, scale)
    with torch.inference_mode():
        for half in period_slices(period):
            for offset in range(STRIDE):
                rows = slice(half.start + offset, half.stop, STRIDE)
                if not len(values[rows]):
                    continue
                batches = torch.from_numpy(tcn_chunks(values[rows], radius))
                logits = torch.cat([model(batch).reshape(-1) for batch in batches.split(32)])
                score[rows] = expit(logits.numpy()[:len(values[rows])])
    assert np.isfinite(score).all() and np.all((0 <= score) & (score <= 1))
    return score


def fit_tcn(spec, matches, train, max_epochs, inner=None):
    torch.manual_seed(TORCH_SEED)
    rng = np.random.default_rng(TORCH_SEED)
    radius = sum(spec["dilations"])
    assert 2 * radius * STRIDE == spec["window_s"] * FPS
    train_x, train_y, mean, scale = make_training_data(matches, train, radius)
    model = build_tcn(spec)
    optimizer_cls = torch.optim.AdamW if spec["optimizer"] == "AdamW" else torch.optim.Adam
    optimizer = optimizer_cls(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_function = nn.BCEWithLogitsLoss(reduction="none")
    best_epoch = max_epochs
    best_score = -np.inf
    stale = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        count = max(1, (len(train_x) + 15) // 16)
        for indices in np.array_split(rng.permutation(len(train_x)), count):
            optimizer.zero_grad(set_to_none=True)
            logits = model(train_x[indices]).squeeze(1)
            target = train_y[indices]
            mask = target >= 0
            loss = (loss_function(logits, target.clamp_min(0)) * mask).sum() / mask.sum()
            loss.backward()
            optimizer.step()

        if inner is None or epoch < MIN_EPOCHS or epoch % CHECK_EVERY:
            continue
        values, labels = inner[:2]
        period = labels["period"].to_numpy()
        score = predict_tcn(model, values, period, mean, scale, radius)
        state = smoothing.viterbi(score, period, VITERBI["penalty"], VITERBI["bias"])
        current = scoring.score_one(state, labels)["alive_spell_f1"]
        history.append(dict(epoch=epoch, inner_f1=float(current)))
        if current > best_score + 1e-9:
            best_score, best_epoch, stale = current, epoch, 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    return (model, mean, scale, radius), best_epoch, history, epoch


def fit_fold(name, spec, held, train, evaluation):
    started = perf_counter()
    held_index = MATCHES.index(held)
    inner = MATCHES[(held_index + 1) % len(MATCHES)]
    inner_matches = [match for match in MATCHES if match not in (held, inner)]
    outer_matches = [match for match in MATCHES if match != held]
    assert len(inner_matches) == 5 and len(outer_matches) == 6 and inner in outer_matches
    selection_start = perf_counter()
    _, selected, history, stopped = fit_tcn(spec, inner_matches, train, MAX_EPOCHS, evaluation[inner])
    selection_s = perf_counter() - selection_start
    refit_start = perf_counter()
    (model, mean, scale, radius), _, _, _ = fit_tcn(spec, outer_matches, train, selected)
    refit_s = perf_counter() - refit_start
    values, labels, geometry, crossings, spells = evaluation[held]
    score = predict_tcn(model, values, labels["period"].to_numpy(), mean, scale, radius)
    scores = score_states(score, labels, geometry, crossings, spells)
    result = dict(
        model=name, match=held, inner_match=inner, selected_epoch=selected, inner_stopped_epoch=stopped,
        selection_s=selection_s, refit_s=refit_s, total_s=perf_counter() - started, history=history, scores=scores,
    )
    print(f"{name} | {held} | epoch {selected}/{stopped} | {result['total_s']:.1f}s | "
          f"F1 {scores[0]['alive_spell_f1']:.4f} / {scores[1]['alive_spell_f1']:.4f}", flush=True)
    return result

## 실행과 요약

시간은 각 후보의 내부 조기 종료 탐색·6경기 재학습·바깥 평가를 모두 합산한다. 공통 입력 준비 시간은 별도다. 이전 기준선 시간에는 조기 종료 탐색이 없다.

In [ ]:
def write_tuning_reports(results, common_s, fingerprint):
    flat = []
    histories = []
    for fold in results:
        fields = {key: value for key, value in fold.items() if key not in ("scores", "history")}
        flat.extend(dict(fields, **score) for score in fold["scores"])
        histories.extend(dict(model=fold["model"], match=fold["match"], inner_match=fold["inner_match"], **row)
                         for row in fold["history"])
    folds = pd.DataFrame(flat)
    expected = {(candidate, match, stage) for candidate in CANDIDATES
                for match in MATCHES for stage in ("viterbi", "line_out")}
    assert set(folds[["model", "match", "stage"]].itertuples(index=False, name=None)) == expected
    assert len(folds) == len(expected) and folds["alive_spell_f1"].between(0, 1).all()
    assert (folds["match"] != folds["inner_match"]).all()
    assert folds["selected_epoch"].between(MIN_EPOCHS, MAX_EPOCHS).all()
    assert (folds["selected_epoch"] <= folds["inner_stopped_epoch"]).all()
    assert (folds["missed_dead"] <= folds["true_dead"]).all()

    rows = []
    prior = pd.read_csv(BASELINE_OUTPUT)
    for row in prior[prior.model.isin(["LightGBM", "TCN 4s"])].to_dict("records"):
        rows.append(dict(model=row["model"] + " (기존)", stage=row["stage"],
                         alive_spell_f1=row["alive_spell_f1"], dead_frame_f1=row["dead_frame_f1"],
                         false_dead=row["false_dead"], missed_dead=row["missed_dead"],
                         selection_s=0, refit_s=row["fit_s"], total_s=row["fold_s"],
                         epochs="6 고정" if row["model"] == "TCN 4s" else "해당 없음"))
    for (model, stage), group in folds.groupby(["model", "stage"], sort=False):
        rows.append(dict(
            model=model, stage=stage, alive_spell_f1=group.alive_spell_f1.mean(),
            dead_frame_f1=group.dead_f1.mean(), false_dead=int(group.false_dead.sum()),
            missed_dead=int(group.missed_dead.sum()), selection_s=group.selection_s.sum(),
            refit_s=group.refit_s.sum(), total_s=group.total_s.sum(),
            epochs=f"{group.selected_epoch.min()}–{group.selected_epoch.max()}",
        ))
    summary = pd.DataFrame(rows)
    folds.to_csv(OUT / "per_match.csv", index=False)
    pd.DataFrame(histories).to_csv(OUT / "inner_epoch_history.csv", index=False)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    previous_manifest = json.loads((BASELINE_OUTPUT.parent / "run_manifest.json").read_text())
    files = ("per_match.csv", "model_summary.csv", "inner_epoch_history.csv")
    manifest = dict(
        run_id=RUN_ID, fingerprints=fingerprint, candidates=CANDIDATES, matches=MATCHES,
        code_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        packages={name: version(name) for name in ("torch", "numpy", "pandas", "scipy", "pyarrow")},
        input_feature_sha256=previous_manifest["input_feature_sha256"], fps=FPS, stride=STRIDE,
        max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS, check_every=CHECK_EVERY, patience=PATIENCE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, torch_seed=TORCH_SEED,
        common_preparation_s=common_s, output_sha256={name: digest(OUT / name) for name in files},
    )
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (OUT / "validation.json").write_text(json.dumps(dict(
        passed=True, expected_rows=len(expected), actual_rows=len(folds), inner_history_rows=len(histories),
    ), indent=2) + "\n", encoding="utf-8")
    print(summary.round(4).to_string(index=False), flush=True)
    return summary


started = perf_counter()
train, _ = prepare_inputs()
evaluation = {match: validation_input(match, BASE) for match in MATCHES}
common_s = perf_counter() - started
fingerprint = {"notebook_sha256": digest(NOTEBOOK), "shared_notebook_sha256": digest(shared),
               "baseline_summary_sha256": digest(BASELINE_OUTPUT)}
checkpoint = OUT / "checkpoint.json"
if checkpoint.is_file():
    state = json.loads(checkpoint.read_text())
    assert state["fingerprints"] == fingerprint
    results = state["results"]
else:
    results = []
completed = {(row["model"], row["match"]) for row in results}
for name, spec in CANDIDATES.items():
    for held in MATCHES:
        if (name, held) in completed:
            continue
        results.append(fit_fold(name, spec, held, train, evaluation))
        pending = checkpoint.with_suffix(".tmp")
        pending.write_text(json.dumps(dict(fingerprints=fingerprint, results=results)), encoding="utf-8")
        pending.replace(checkpoint)
summary = write_tuning_reports(results, common_s, fingerprint)